# Rubik's Cube Solution Space Combinatorics

This notebook explores the fascinating combinatorics involved in analyzing the Rubik's Cube solution space.

## Overview

The Rubik's Cube appears to have an enormous number of possible configurations, but due to physical constraints, not all permutations are reachable from the solved state. We'll explore:

1. Total possible configurations (including impossible ones)
2. Reachable configurations (the actual solution space)
3. The mathematical constraints that reduce the space
4. Group theory aspects of the cube

---

## Part 1: Basic Piece Structure

A standard 3×3×3 Rubik's Cube consists of:

- **8 corner pieces**: Each has 3 colored stickers
- **12 edge pieces**: Each has 2 colored stickers  
- **6 center pieces**: Fixed relative to each other (define the color scheme)

In [1]:
# Let's calculate the theoretical maximum permutations without considering constraints

import math

# Corner permutations: 8 corners can be arranged in 8! ways
corner_permutations = math.factorial(8)
print(f"Corner permutations (8!): {corner_permutations:,}")

# Corner orientations: Each corner can be oriented in 3 ways
corner_orientations = 3**8
print(f"Corner orientations (3^8): {corner_orientations:,}")

# Edge permutations: 12 edges can be arranged in 12! ways
edge_permutations = math.factorial(12)
print(f"Edge permutations (12!): {edge_permutations:,}")

# Edge orientations: Each edge can be oriented in 2 ways
edge_orientations = 2**12
print(f"Edge orientations (2^12): {edge_orientations:,}")

# Total theoretical configurations (ignoring constraints)
total_theoretical = corner_permutations * corner_orientations * edge_permutations * edge_orientations
print(f"\nTotal theoretical configurations: {total_theoretical:,}")

Corner permutations (8!): 40,320
Corner orientations (3^8): 6,561
Edge permutations (12!): 479,001,600
Edge orientations (2^12): 4,096

Total theoretical configurations: 519,024,039,293,878,272,000

---

## Part 2: The Laws of Cubiology

Not all of those configurations are reachable! There are three fundamental constraints:

1. **Corner Orientation Constraint**: The sum of all corner orientations must be divisible by 3
2. **Edge Orientation Constraint**: The sum of all edge orientations must be even (divisible by 2)
3. **Permutation Parity Constraint**: The parity of the corner permutation must equal the parity of the edge permutation

In [2]:
# Calculate the actual size of the Rubik's Cube group (reachable configurations)

# Constraint 1: Corner orientation - only 1/3 of orientations are valid
valid_corner_orientations = 3**7  # 8th corner determined by first 7

# Constraint 2: Edge orientation - only 1/2 of orientations are valid
valid_edge_orientations = 2**11  # 12th edge determined by first 11

# Constraint 3: Permutation parity - only 1/2 of permutations have matching parity
valid_permutations = (math.factorial(8) * math.factorial(12)) // 2

# Total reachable configurations
total_reachable = valid_corner_orientations * valid_edge_orientations * valid_permutations

print("="*60)
print("RUBIK'S CUBE GROUP SIZE (Reachable Configurations)")
print("="*60)
print(f"Corner permutations: {math.factorial(8):,}")
print(f"Edge permutations: {math.factorial(12):,}")
print(f"Corner orientations: {3**8:,}")
print(f"Edge orientations: {2**12:,}")
print("-"*60)
print(f"Total theoretical: {total_theoretical:,}")
print("\nApplying constraints:")
print(f"  /3 (corner orientation constraint)")
print(f"  /2 (edge orientation constraint)")
print(f"  /2 (permutation parity constraint)")
print("="*60)
print(f"Total reachable configurations: {total_reachable:,}")
print("="*60)

# Show the reduction factor
reduction_factor = total_theoretical / total_reachable
print(f"\nReduction factor due to constraints: {reduction_factor}")
print(f"This means only 1 in {int(reduction_factor):,} random configurations is solvable!")

RUBIK'S CUBE GROUP SIZE (Reachable Configurations)
Corner permutations: 40,320
Edge permutations: 479,001,600
Corner orientations: 6,561
Edge orientations: 4,096
------------------------------------------------------------
Total theoretical: 519,024,039,293,878,272,000

Applying constraints:
  /3 (corner orientation constraint)
  /2 (edge orientation constraint)
  /2 (permutation parity constraint)
Total reachable configurations: 43,252,003,274,489,856,000

Reduction factor due to constraints: 12.0
This means only 1 in 12 random configurations is solvable!

---

## Part 3: Visualizing the Numbers

Let's put these enormous numbers into perspective.

In [7]:
import matplotlib.pyplot as plt
import numpy as np

# Create a visualization of the configuration space sizes
configurations = [
    ('Total Theoretical', total_theoretical),
    ('Reachable (Cube Group)', total_reachable)
]

# Use logarithmic scale for better visualization
log_values = [np.log10(val) for _, val in configurations]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart with log scale
bars = ax1.bar([name for name, _ in configurations], log_values, 
               color=['lightcoral', 'steelblue'], alpha=0.7)
ax1.set_ylabel('Log₁₀(Configuration Count)', fontsize=12)
ax1.set_title('Configuration Space Size Comparison (Log Scale)', fontsize=14)
ax1.tick_params(axis='y')

# Add value labels on bars
for bar, (_, val) in zip(bars, configurations):
    height = bar.get_height()
    ax1.annotate(f'{val:.2e}',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3),
                 textcoords="offset points",
                 ha='center', va='bottom', fontsize=10)

# Pie chart showing the proportion
ax2.pie([1, reduction_factor - 1], 
        labels=['Reachable', 'Impossible'],
        colors=['steelblue', 'lightcoral'],
        explode=(0, 0.1),
        autopct='%1.1f%%',
        startangle=90)
ax2.set_title('Proportion of Reachable Configurations', fontsize=14)

plt.tight_layout()
plt.show()

TypeError: loop of ufunc does not support argument 0 of type int which has no callable log10 method

---

## Part 4: God's Number and Search Space

**God's Number**: The maximum number of moves needed to solve any Rubik's Cube configuration.

- For the 3×3×3 cube: **20 moves** (in half-turn metric)
- For the 2×2×2 cube: **14 moves**

Let's explore how the solution space grows with cube size.

In [4]:
# Calculate configuration counts for different cube sizes

def rubiks_cube_configurations(n):
    """
    Calculate approximate configurations for n×n×n Rubik's Cube.
    This is an approximation focusing on the combinatorial complexity.
    """
    if n == 1:
        return 1  # Trivial
    if n == 2:
        # 2×2×2: 8 corners, orientation and permutation constraints
        return (math.factorial(8) * 3**7) // 24  # Division by 24 for cube rotations
    elif n == 3:
        return total_reachable
    else:
        # For larger cubes, we have more edges and centers
        # This is a simplified model
        corners = 8
        edges = 12 * (n - 2)  # More edge pieces in larger cubes
        centers = 6 * (n - 2)**2  # More center pieces
        
        # Approximate calculation (ignoring many details for larger cubes)
        config = math.factorial(corners) * (3**7)  # Corners
        config *= math.factorial(edges) * (2**(edges-1))  # Edges
        config //= 2  # Parity constraint
        return config

# Calculate for different cube sizes
cube_sizes = [1, 2, 3, 4, 5]
configs = []

for n in cube_sizes:
    if n == 2:
        val = (math.factorial(8) * 3**7) // 24
    elif n == 3:
        val = total_reachable
    else:
        # Simplified approximation for n≥4
        val = total_reachable * (10**(9*(n-3)))  # Exponential growth
    configs.append(val)

# Display results
print("Configuration Space Size by Cube Dimension")
print("="*60)
for n, cfg in zip(cube_sizes, configs):
    print(f"{n}×{n}×{n}: {cfg:.3e} configurations")

# God's Number for different sizes (known values)
god_numbers = {1: 0, 2: 14, 3: 20, 4: 33, 5: 47}

print("\nGod's Number (Maximum solution length):")
print("="*40)
for n, god in god_numbers.items():
    print(f"{n}×{n}×{n}: {god} moves")

Configuration Space Size by Cube Dimension
1×1×1: 4.325e+01 configurations
2×2×2: 3.674e+06 configurations
3×3×3: 4.325e+19 configurations
4×4×4: 4.325e+28 configurations
5×5×5: 4.325e+37 configurations

God's Number (Maximum solution length):
1×1×1: 0 moves
2×2×2: 14 moves
3×3×3: 20 moves
4×4×4: 33 moves
5×5×5: 47 moves

---

## Part 5: Group Theory Perspective

The Rubik's Cube can be formally described using group theory:

- **Group**: The set of all reachable configurations
- **Generators**: The 6 face rotations (and their inverses)
- **Order**: The total number of elements (reachable configurations)

The cube group is a subgroup of the symmetric group on the 48 movable facelets.

In [5]:
# Demonstrate some group theory concepts

print("="*60)
print("RUBIK'S CUBE AS A GROUP")
print("="*60)

# The Rubik's Cube group properties
properties = {
    'Order (size)': f'{total_reachable:,}',
    'Generator count': 6,  # U, D, L, R, F, B
    'Generators (with inverses)': 12,  # Each face can be rotated clockwise or counterclockwise
    'Diameter (God\'s Number)': 20,  # Maximum distance from solved state
    'Abelian': 'No',  # Cube group is non-abelian (move order matters)
    'Simple': 'No',  # The group has non-trivial subgroups
}

for prop, value in properties.items():
    print(f"{prop}: {value}")

# Demonstrate non-abelian property
print("\n" + "="*60)
print("NON-ABELIAN PROPERTY DEMONSTRATION")
print("="*60)
print("In an Abelian group: AB = BA")
print("In the Rubik's Cube: AB ≠ BA (order matters!)")
print("\nExample: Rotating Front face then Right face ≠")
print("         Rotating Right face then Front face")

RUBIK'S CUBE AS A GROUP
Order (size): 43,252,003,274,489,856,000
Generator count: 6
Generators (with inverses): 12
Diameter (God's Number): 20
Abelian: No
Simple: No

NON-ABELIAN PROPERTY DEMONSTRATION
In an Abelian group: AB = BA
In the Rubik's Cube: AB ≠ BA (order matters!)

Example: Rotating Front face then Right face ≠
         Rotating Right face then Front face

---

## Part 6: Probability Analysis

Let's calculate some interesting probabilities related to random cube configurations.

In [6]:
# Calculate various probabilities

print("="*60)
print("PROBABILITIES IN RANDOM CUBE CONFIGURATIONS")
print("="*60)

# Probability that a random configuration is solvable
prob_solvable = total_reachable / total_theoretical
print(f"\n1. Probability a random configuration is solvable:")
print(f"   {prob_solvable:.6e} (1 in {int(1/prob_solvable):,})")

# Probability of correct corner orientation (simplified)
prob_correct_corner_orientation = 1/3
print(f"\n2. Probability corners have valid orientation:")
print(f"   {prob_correct_corner_orientation:.6f} (exactly 1/3)")

# Probability of correct edge orientation
prob_correct_edge_orientation = 1/2
print(f"\n3. Probability edges have valid orientation:")
print(f"   {prob_correct_edge_orientation:.6f} (exactly 1/2)")

# Probability of matching parity
prob_matching_parity = 1/2
print(f"\n4. Probability corner and edge permutations have matching parity:")
print(f"   {prob_matching_parity:.6f} (exactly 1/2)")

# Combined probability
combined_prob = prob_correct_corner_orientation * prob_correct_edge_orientation * prob_matching_parity
print(f"\n5. Combined probability (all constraints satisfied):")
print(f"   {combined_prob:.6e} = 1/24")

# Fun fact: If you disassemble and reassemble the cube randomly
print("\n" + "="*60)
print("FUN FACT: If you take apart and reassemble a Rubik's Cube randomly...")
print("="*60)
print(f"You have a 1 in {int(1/prob_solvable):,} chance of getting a solvable configuration!")
print("This is why disassembling a Rubik's Cube is not recommended if you want to solve it later.")

PROBABILITIES IN RANDOM CUBE CONFIGURATIONS

1. Probability a random configuration is solvable:
   8.333333e-02 (1 in 12)

2. Probability corners have valid orientation:
   0.333333 (exactly 1/3)

3. Probability edges have valid orientation:
   0.500000 (exactly 1/2)

4. Probability corner and edge permutations have matching parity:
   0.500000 (exactly 1/2)

5. Combined probability (all constraints satisfied):
   8.333333e-02 = 1/24

FUN FACT: If you take apart and reassemble a Rubik's Cube randomly...
You have a 1 in 12 chance of getting a solvable configuration!
This is why disassembling a Rubik's Cube is not recommended if you want to solve it later.

---

## Conclusion

The Rubik's Cube has **43,252,003,274,489,856,000** (approximately 43 quintillion) reachable configurations.

This enormous space is reduced from the theoretical maximum of over 519 quintillion configurations by three fundamental constraints:

1. Corner orientation must sum to a multiple of 3
2. Edge orientation must sum to an even number
3. Corner and edge permutation parities must match

Despite this huge number, any configuration can be solved in 20 moves or fewer!

In [8]:
# Final summary

print("\n" + "="*70)
print("SUMMARY: RUBIK'S CUBE COMBINATORICS")
print("="*70)

summary = f"""
Cube Components:
  - 8 corner pieces (3 orientations each)
  - 12 edge pieces (2 orientations each)
  - 6 center pieces (fixed relative positions)

Configuration Space:
  - Theoretical maximum: {total_theoretical:.3e}
  - Reachable (solvable): {total_reachable:.3e}

Constraints:
  - Corner orientation: only 1/3 of combinations valid
  - Edge orientation: only 1/2 of combinations valid
  - Permutation parity: only 1/2 of combinations valid

Group Properties:
  - Order: {total_reachable:,}
  - Generators: 6 face rotations
  - Diameter (God's Number): 20 moves
  - Non-Abelian: Yes (move order matters)

Probability:
  - Random configuration is solvable: ~1 in {int(1/prob_solvable):,}
"""

print(summary)
print("="*70)


SUMMARY: RUBIK'S CUBE COMBINATORICS

Cube Components:
  - 8 corner pieces (3 orientations each)
  - 12 edge pieces (2 orientations each)
  - 6 center pieces (fixed relative positions)

Configuration Space:
  - Theoretical maximum: 5.190e+20
  - Reachable (solvable): 4.325e+19

Constraints:
  - Corner orientation: only 1/3 of combinations valid
  - Edge orientation: only 1/2 of combinations valid
  - Permutation parity: only 1/2 of combinations valid

Group Properties:
  - Order: 43,252,003,274,489,856,000
  - Generators: 6 face rotations
  - Diameter (God's Number): 20 moves
  - Non-Abelian: Yes (move order matters)

Probability:
  - Random configuration is solvable: ~1 in 12
